## Micrograd

In [2]:
import math

class Value:

    def __init__(self, data, _children = (), op = (), label = " "):
        self.data = data
        self.grad = 0
        self._backward = lambda: None 
        self._prev = set(_children)
        self._op = op
        self._label = label
    
    def __repr__(self):
        return f"Value(label={self._label}, data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        
        out._backward = _backward 
        return out

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backward 
        return out

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * (other**-1)

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int and float powers are supported for now"
        out = Value(self.data ** other, (self,), f"**{other}")

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1)/(math.exp(2*n) + 1)
        out = Value(t, (self, ), label="tanh")

        def _backward():
            self.grad +=  (1 - t**2) * out.grad
        
        out._backward = _backward 
        return out

    def backprop(self):
        self._backward()
        # print(f"label: {self._label}, grad: {self.grad}")
        for child in self._prev:
            child.backprop()

In [3]:
a = Value(3.0, label="a")
b = a + a
b.grad = 1
b.backprop()

In [4]:
a + 1
a * 2
2 * a
2 + a

Value(label= , data=5.0)

In [5]:
x1 = Value(2.0, label="x1")
x2 = Value(0.0, label="x2")

w1 = Value(-3.0, label="w1")
w2 = Value(1.0, label="w2")

b = Value(6.8813735870195432, label="b")

x1w1 = x1*w1
x1w1._label = "x1*w1"

x2w2 = x2*w2
x2w2._label = "x2*w2"

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2._label = "w1*w1 + x2*w2"

n = x1w1x2w2 + b
n._label = "n"

o = n.tanh()
o._label = "o"

In [6]:
o.grad = 1.0

In [7]:
o.backprop()

In [8]:
import random

class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for i in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        assert len(x) == len(self.w), "number of inputs is not equal to the number of weights"
        act = sum([xi * self.w[i] for i, xi in enumerate(x)]) + self.b
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

In [9]:
x = [2.0, 3.0]
n = Neuron(len(x))
n(x)

Value(label=tanh, data=0.9947923349982284)

In [10]:
class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        out = [neuron(x) for neuron in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

In [11]:
x = [2.0, 3.0]
la = Layer(len(x), 3)
la(x)

[Value(label=tanh, data=-0.9964296821332301),
 Value(label=tanh, data=0.6262165068446033),
 Value(label=tanh, data=0.6085567850260301)]

In [12]:
class MLP:

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        out = x
        for layer in self.layers:
            out = layer(out)
        return out

    def parameters(self):
        return [p for la in self.layers for p in la.parameters()]

In [13]:
x = [2.0, 3.0, -1.0]
mlp = MLP(len(x), [4, 4, 1])
mlp(x)

Value(label=tanh, data=0.02866329190255835)

In [14]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0]

In [15]:
ypreds = [mlp(x) for x in xs]
ypreds

[Value(label=tanh, data=0.02866329190255835),
 Value(label=tanh, data=0.37709666285859733),
 Value(label=tanh, data=0.02448958091843014),
 Value(label=tanh, data=0.03986627860322582)]

In [16]:
loss = sum([(ypred - y)**2 for y, ypred in zip(ys, ypreds)])
loss._label = "loss"
loss

Value(label=loss, data=4.811325883727499)

In [17]:
loss.grad = 1.0
loss.backprop()

In [18]:
len(mlp.parameters())

41

In [19]:
STEP = 0.01

for p in mlp.parameters():
    p_old = p.data
    p.data += p.grad * (-STEP)
    print(f"parameter updated from {p_old} to {p.data}")

parameter updated from -0.6735037221028126 to -0.8808051377391148
parameter updated from 0.3185358228350792 to 0.11586048551144404
parameter updated from -0.21721053579118998 to -0.07576506134921504
parameter updated from -0.7566521120084857 to -0.778438888019704
parameter updated from 0.4405124135292282 to 0.4842517274144765
parameter updated from 0.7121048047752727 to 0.7335598399285671
parameter updated from 0.8503514479113463 to 0.834995613021366
parameter updated from 0.8114634753845291 to 0.817434863697066
parameter updated from 0.3303612371190614 to 0.8501603485503176
parameter updated from -0.8632294011510628 to -0.8899656686553875
parameter updated from -0.6735311064143166 to -0.6699260755369518
parameter updated from -0.8062393313976608 to -0.7759514620337414
parameter updated from -0.3969740148271441 to -0.2713955058582318
parameter updated from 0.6492047860373571 to 0.2757568466551863
parameter updated from 0.436598466813541 to 0.6345202591038555
parameter updated from 0.99

In [20]:
TRAINING_ITER = 100

def train(mlp, xs, ys):
    for i in range(TRAINING_ITER):
        # forward
        ypreds = [mlp(x) for x in xs]
        loss = sum([(ypred - y)**2 for y, ypred in zip(ys, ypreds)])
        print(f"loss {loss}")
        loss._label = "loss"
        loss.grad = 1
        for p in mlp.parameters():
            p.grad = 0.0
        # backward
        loss.backprop()
        # update
        for p in mlp.parameters():
            p.data += p.grad * (-STEP)
    

In [21]:
train(mlp, xs, ys)

ypreds = [mlp(x) for x in xs]
loss = sum([(ypred - y)**2 for y, ypred in zip(ys, ypreds)])
loss

loss Value(label= , data=3.1109558630069056)
loss Value(label= , data=1.971276592286253)
loss Value(label= , data=1.541780891814242)
loss Value(label= , data=1.2988233711877117)
loss Value(label= , data=1.0960036720360005)
loss Value(label= , data=0.8835697412829031)
loss Value(label= , data=0.7477167176636879)
loss Value(label= , data=0.6781653178295719)
loss Value(label= , data=0.5716321976915862)
loss Value(label= , data=0.5566539535193485)
loss Value(label= , data=0.4767626918909578)
loss Value(label= , data=0.41236767796668095)
loss Value(label= , data=0.3605941131170672)
loss Value(label= , data=0.33265321475635834)
loss Value(label= , data=0.31766093237320503)
loss Value(label= , data=0.29152806368742856)
loss Value(label= , data=0.2733715528873797)
loss Value(label= , data=0.2655658567511464)
loss Value(label= , data=0.24204106074858117)
loss Value(label= , data=0.22731982630966854)
loss Value(label= , data=0.21377346241736947)
loss Value(label= , data=0.20237916320247987)
loss

Value(label= , data=0.03725199874815649)

In [22]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

In [23]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0]


In [24]:
from lib.mlp import MLP

mlp = MLP(len(x), [4, 4, 1])
mlp.train(xs, ys, 100, 0.01)